# TRACE-GRAG — Reproducibility Notebook

**Lexical, Semantic, and Graph Retrieval for Repository-Level Bug Localization under Local Gemma-Class Constraints.**

Companion to the Gemma 4 Developer Agent *Paper Track* writeup. This notebook (1) builds the provenance-disciplined code graph on a bundled fixture (no network, no Ollama), (2) runs the lexical (E1) and graph (E3) retrievers, and (3) renders the headline 24-task results and chart from the committed evaluation run. Every number traces to `results/summary.json`.

Repo (Apache-2.0): https://github.com/bobby-nandigam/trace-grag

In [ ]:
# Clone the public repo and install the package (needs internet on in the notebook settings).
import os, subprocess, sys
if not os.path.isdir('trace-grag'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/bobby-nandigam/trace-grag.git'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'trace-grag'], check=True)
os.chdir('trace-grag')
print('cwd:', os.getcwd())

## 1. Provenance-disciplined code graph

A two-pass `ast` parser emits 6 node types and 6 edge types. **Every edge records how it was derived (`provenance`) and a calibrated confidence; unresolvable call/import targets are marked `unresolved` rather than invented.**

In [ ]:
from trace_grag import PythonParser, RepoGraph
nodes, edges = PythonParser('data/sample/minirepo', repo_name='minirepo').parse()
g = RepoGraph('minirepo'); g.add_nodes(nodes); g.add_edges(edges)
stats = g.stats()
print('nodes by type:', stats['nodes_by_type'])
print('edges by type:', stats['edges_by_type'])
print('unresolved targets (recorded, not invented):', stats['unresolved_targets'])

In [ ]:
# Show that unresolved CALLS edges are marked explicitly, with their raw target preserved.
for u, v, k, d in g.graph.edges(keys=True, data=True):
    if k == 'CALLS' and d['provenance'] == 'unresolved':
        print(f"UNRESOLVED call -> {d['raw_target']!r}  (confidence {d['confidence']})")
        break
# ...and a resolved one, for contrast
for u, v, k, d in g.graph.edges(keys=True, data=True):
    if k == 'CALLS' and d['provenance'] == 'call_resolved':
        print(f"RESOLVED  call -> {g.node(v).qualified_name}  (confidence {d['confidence']})")
        break

## 2. Retrieval baselines (E1 lexical, E3 graph)

Both run with no external services. (E2 semantic needs a local Ollama serving `nomic-embed-text`; see the repo's `README`.)

In [ ]:
from trace_grag.retrieval.lexical import BM25Retriever
from trace_grag.retrieval.graph_retrieval import GraphRetriever
issue = 'divide raises an error on zero division in the calculator'
for R in (BM25Retriever(), GraphRetriever()):
    ranked = R.rank(issue, g)
    top = [(g.node(i).qualified_name, round(s, 3)) for i, s in ranked[:3]]
    print(f'{R.name:7s} top-3:', top)

## 3. Headline results — 24 SWE-bench Lite tasks

Loaded from the committed `results/summary.json` (run `run-20261112-103442`, 0 errors, E1/E2/E3 on every task). Values are mean [95% bootstrap CI].

In [ ]:
import json, pandas as pd
summary = json.load(open('results/summary.json'))
table = summary['summaries']['all']['table']
metrics = ['hit@1', 'hit@5', 'hit@10', 'recall@10', 'mrr']
rows = {}
for strat in ['bm25', 'semantic', 'graph']:
    rows[strat] = {m: f"{table[strat][m][0]:.3f} [{table[strat][m][1]:.3f},{table[strat][m][2]:.3f}]" for m in metrics}
    rows[strat]['lat_s'] = round(table[strat]['latency_s'][0], 2)
df = pd.DataFrame(rows).T
df.index = ['E1 BM25', 'E2 Semantic', 'E3 Graph']
df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
strats = ['bm25', 'semantic', 'graph']; labels = ['E1 BM25', 'E2 Semantic', 'E3 Graph']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, metric in zip(ax, ['hit@1', 'mrr']):
    means = [table[s][metric][0] for s in strats]
    lo = [table[s][metric][0] - table[s][metric][1] for s in strats]
    hi = [table[s][metric][2] - table[s][metric][0] for s in strats]
    a.bar(labels, means, yerr=[lo, hi], capsize=6, color=['#4C78A8', '#F58518', '#54A24B'])
    a.set_title(f'{metric}  (mean, 95% bootstrap CI)'); a.set_ylim(0, 1)
plt.suptitle('Localization on 24 SWE-bench Lite tasks — CIs overlap: no significant winner')
plt.tight_layout(); plt.show()

## 4. Takeaways

- **F4 (headline)** — retrieval **recall gates the agent**: semantic's higher recall@10 (0.71) gives Gemma 4 its best localization (0.46 vs 0.38).
- **F5** — **Gemma reranking nearly doubles top-1** over the retriever's own rank-1 (BM25 0.21 → 0.38): a small local model reading 10 candidates repairs weak ranking.
- **F6** — given gold is in the top-10, Gemma picks it ~64% **regardless of retriever**; graph expansion (E3) = BM25 (E1) exactly, for retrieval and the agent.

**Implication:** for on-device SE agents, spend the budget on cheap high-**recall** retrieval and let the model rerank — not on graph expansion. Full method, CIs, and discussion in the writeup; all code, manifest, and harness in the released repo (Apache-2.0).

In [ ]:
gem = json.load(open('results/gemma_localization/summary.json'))['strategies']
gdf = pd.DataFrame({k: gem[k] for k in ['bm25', 'semantic', 'graph']}).T
gdf = gdf[['gemma_hit@1', 'gold_in_top10', 'retr_top1', 'gemma_given_gold_in_top10']]
gdf.index = ['E1 BM25', 'E2 Semantic', 'E3 Graph']
print('Gemma 4 E2B localization (rate over 24 tasks):')
gdf

## 4. Takeaways

- **F1** — graph expansion (E3) ties BM25 (E1) in aggregate; it reshuffles per-task rankings but nets to zero.
- **F2** — semantic (E2) is *complementary*: higher hit@1/MRR, recovers some repos lexical misses (xarray, astropy), regresses on others (scikit-learn, pylint); **all CIs overlap → no significant winner at n=24**.
- **F3** — semantic costs ~500× more latency locally (median 47 s vs 0.09 s).

**Implication:** for on-device SE agents, cheap lexical retrieval is a strong default; lexical–semantic *fusion* is the promising direction. Full method and discussion in the writeup; full code, manifest, and harness in the released repo (Apache-2.0).